In [1]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import time
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [2]:
telco_data = pd.read_csv("../data/processed/feature_engineered_customers.csv")

In [3]:
x = telco_data.drop(columns=["Churn", "customerID"])
y = telco_data["Churn"].map({"No": 0, "Yes": 1})

In [4]:
categorical_features = x.select_dtypes(include=["object", "category"]).columns.tolist()
categorical_features

C:\Users\SURYA\AppData\Local\Temp\ipykernel_17416\1447111206.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = x.select_dtypes(include=["object", "category"]).columns.tolist()


['gender',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'TenureGroup',
 'MonthlyChargesGroup',
 'TotalChargesGroup',
 'InternetTechSupport']

In [5]:
x_encoded = pd.get_dummies(
    x,
    columns=categorical_features,
    drop_first=False,
    dtype=int
)
x_encoded.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,NumberOfServices,IsDSL,IsFiberOptic,NoInternet,ChargeToTenureRatio,HighValueCustomer,...,TenureGroup_Very Long Term,MonthlyChargesGroup_High,MonthlyChargesGroup_Low,MonthlyChargesGroup_Medium,TotalChargesGroup_High,TotalChargesGroup_Low,TotalChargesGroup_Medium,InternetTechSupport_Internet_With_Support,InternetTechSupport_Internet_Without_Support,InternetTechSupport_No_Internet
0,0,1,29.85,29.85,2,1,0,0,29.850000,0,...,0,0,1,0,0,1,0,0,1,0
1,0,34,56.95,1889.50,3,1,0,0,55.573529,0,...,0,0,0,1,0,0,1,0,1,0
2,0,2,53.85,108.15,3,1,0,0,54.075000,0,...,0,0,0,1,0,1,0,0,1,0
3,0,45,42.30,1840.75,4,1,0,0,40.905556,0,...,0,0,0,1,0,0,1,1,0,0
4,0,2,70.70,151.65,1,0,1,0,75.825000,0,...,0,1,0,0,0,1,0,0,1,0


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    x_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (5634, 66)
X_test shape: (1409, 66)


In [7]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_features

['SeniorCitizen',
 'tenure',
 'MonthlyCharges',
 'TotalCharges',
 'NumberOfServices',
 'IsDSL',
 'IsFiberOptic',
 'NoInternet',
 'ChargeToTenureRatio',
 'HighValueCustomer',
 'HighRiskCustomer',
 'gender_Female',
 'gender_Male',
 'Partner_No',
 'Partner_Yes',
 'Dependents_No',
 'Dependents_Yes',
 'PhoneService_No',
 'PhoneService_Yes',
 'MultipleLines_No',
 'MultipleLines_No phone service',
 'MultipleLines_Yes',
 'InternetService_DSL',
 'InternetService_Fiber optic',
 'InternetService_No',
 'OnlineSecurity_No',
 'OnlineSecurity_No internet service',
 'OnlineSecurity_Yes',
 'OnlineBackup_No',
 'OnlineBackup_No internet service',
 'OnlineBackup_Yes',
 'DeviceProtection_No',
 'DeviceProtection_No internet service',
 'DeviceProtection_Yes',
 'TechSupport_No',
 'TechSupport_No internet service',
 'TechSupport_Yes',
 'StreamingTV_No',
 'StreamingTV_No internet service',
 'StreamingTV_Yes',
 'StreamingMovies_No',
 'StreamingMovies_No internet service',
 'StreamingMovies_Yes',
 'Contract_Month

In [8]:
scaler = StandardScaler()

X_train[numeric_features] = scaler.fit_transform(
    X_train[numeric_features]
)

X_test[numeric_features] = scaler.transform(
    X_test[numeric_features]
)

print("Numerical scaled")

Numerical scaled


In [9]:
ann_model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.3),

    Dense(32, activation="relu"),
    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
ann_model.summary()

C:\AximSoft_Weekly_Tasks\week_12\customer_env\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape           ┃      Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ dense (Dense)                 │ (None, 64)             │        4,288 │
├───────────────────────────────┼────────────────────────┼──────────────┤
│ dropout (Dropout)             │ (None, 64)             │            0 │
├───────────────────────────────┼────────────────────────┼──────────────┤
│ dense_1 (Dense)               │ (None, 32)             │        2,080 │
├───────────────────────────────┼────────────────────────┼──────────────┤
│ dropout_1 (Dropout)           │ (None, 32)             │            0 │
├───────────────────────────────┼────────────────────────┼──────────────┤
│ dense_2 (Dense)               │ (None, 1)              │           33 │
└───────────────────────────────┴────────────────────────┴──────────────┘

 Total params: 6,401 (25.00 KB)

 Trainable params: 6,401 (25.00 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

start_time = time.time()

history = ann_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

ann_training_time = time.time() - start_time

print(f"ANN Training Time: {ann_training_time:.4f} seconds")

Epoch 1/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.7630 - loss: 0.4893 - val_accuracy: 0.7950 - val_loss: 0.4501
Epoch 2/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7872 - loss: 0.4537 - val_accuracy: 0.7924 - val_loss: 0.4413
Epoch 3/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8065 - loss: 0.4297 - val_accuracy: 0.7941 - val_loss: 0.4415
Epoch 4/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8061 - loss: 0.4240 - val_accuracy: 0.7906 - val_loss: 0.4380
Epoch 5/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8056 - loss: 0.4265 - val_accuracy: 0.7941 - val_loss: 0.4394
Epoch 6/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8043 - loss: 0.4199 - val_accuracy: 0.7950 - val_loss: 0.4413
ANN Training Time: 10.4097 seconds


In [12]:
# Measure inference time
start_time = time.time()

ann_probabilities = ann_model.predict(X_test, verbose=0).ravel()

ann_inference_time = time.time() - start_time

# Convert probabilities to class predictions
ann_predictions = (ann_probabilities >= 0.5).astype(int)

# Metrics
ann_accuracy = accuracy_score(y_test, ann_predictions)
ann_precision = precision_score(y_test, ann_predictions)
ann_recall = recall_score(y_test, ann_predictions)
ann_f1 = f1_score(y_test, ann_predictions)
ann_roc_auc = roc_auc_score(y_test, ann_probabilities)

print("ANN Results")

print(f"Accuracy       : {ann_accuracy:.4f}")
print(f"Precision      : {ann_precision:.4f}")
print(f"Recall         : {ann_recall:.4f}")
print(f"F1 Score       : {ann_f1:.4f}")
print(f"ROC-AUC        : {ann_roc_auc:.4f}")
print(f"Training Time  : {ann_training_time:.4f} sec")
print(f"Inference Time : {ann_inference_time:.4f} sec")

ANN Results
Accuracy       : 0.7928
Precision      : 0.6367
Recall         : 0.5107
F1 Score       : 0.5668
ROC-AUC        : 0.8370
Training Time  : 10.4097 sec
Inference Time : 0.5151 sec


In [13]:
ann_model.save("../models/small_ann_model.keras")
print("saved")

saved
